# 콘텐츠 기반 필터링 따라가기

팀원이 만들어 둔 **영화 장르 벡터**를 가져와서,
학습/테스트만 나눈 뒤 추천까지 갑니다.

- 가져오는 것: `data/processed/movies_multihot_data.npz` (영화 정보)
- 가져오지 않는 것: `user_profiles_data.npz` (사람 취향. 시험 평점까지 들어가 있음)
- 사람 취향은 **학습 평점만**으로 여기서 다시 만듭니다.


## 0. 라이브러리와 경로


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if (ROOT / "src").exists() is False:
    ROOT = ROOT.parent

RAW_DIR = ROOT / "preprocessing_and_data" / "movielens-1m"
PROCESSED = ROOT / "data" / "processed"

print("프로젝트 루트:", ROOT)
print("영화 npz:", (PROCESSED / "movies_multihot_data.npz").exists())
print("평점 원본:", (RAW_DIR / "ratings.dat").exists())


프로젝트 루트: c:\Users\MYCOM\Desktop\학교\4-1\머신러닝\project\machine_learning_term_project
영화 npz: True
평점 원본: True


## 1. 팀원이 만든 영화 벡터 가져오기

`preprocess_content_based.py`를 돌리면 `data/processed/movies_multihot_data.npz`가 생깁니다.

들어 있는 것:
- `movie_ids`, `titles`: 영화 번호와 제목
- `classes`: 장르 이름
- `multi_hot_norm`: 영화마다 길이 1로 맞춘 장르 벡터 (예측에 사용)

프로젝트에서는 영화 번호를 `item_id`라고 부릅니다.


In [2]:
movies_npz = np.load(PROCESSED / "movies_multihot_data.npz", allow_pickle=True)
print("키:", movies_npz.files)

item_ids = movies_npz["movie_ids"]
titles = movies_npz["titles"]
genre_cols = list(movies_npz["classes"])
movie_multihot_norm = movies_npz["multi_hot_norm"]

item_index = {int(i): k for k, i in enumerate(item_ids)}

print("영화 수, 장르 수:", movie_multihot_norm.shape)
print("장르:", genre_cols)
pd.DataFrame({"item_id": item_ids, "title": titles}).head()


키: ['multi_hot_norm', 'classes', 'movie_ids', 'titles']
영화 수, 장르 수: (3883, 18)
장르: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,item_id,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)


In [25]:
movie_multihot_norm_df = pd.DataFrame(
    movie_multihot_norm,
    index=item_ids,
    columns=genre_cols,
)
movie_multihot_norm_df.index.name = "item_id"
movie_multihot_norm_df.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
item_id,,,,,,,,,,,,,,,,,,
1,0.0,0.00000,0.57735,0.57735,0.577350,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.57735,0.00000,0.57735,0.000000,0.0,0.0,0.000000,0.57735,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.00000,0.00000,0.00000,0.707107,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.707107,0.0,0.0,0.0,0.0
4,0.0,0.00000,0.00000,0.00000,0.707107,0.0,0.0,0.707107,0.00000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
5,0.0,0.00000,0.00000,0.00000,1.000000,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


사람 취향을 만들 때는 0/1 장르 표가 필요합니다.
정규화 벡터에서 0이 아닌 칸이 그 영화의 장르입니다.


In [11]:
movie_multihot = (movie_multihot_norm > 1e-12).astype(int)
movie_feature_df = pd.DataFrame(
    movie_multihot,
    index=item_ids,
    columns=genre_cols,
)
print("첫 영화:", titles[0])
print(movie_feature_df.iloc[0][movie_feature_df.iloc[0] == 1])
movie_feature_df.head()


첫 영화: Toy Story (1995)
Animation     1
Children's    1
Comedy        1
Name: 1, dtype: int64


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2. 평점 읽기

추천하려면 누가 몇 점을 줬는지도 필요합니다. 이건 원본 `ratings.dat`을 씁니다.


In [12]:
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    encoding="ISO-8859-1",
    names=["user_id", "item_id", "rating", "timestamp"],
)
print(ratings.shape)
print(ratings["rating"].value_counts().sort_index())
ratings.head()


(1000209, 4)
rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64


,user_id,item_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## 3. 학습/테스트 나누기

사용자마다 평점의 일부를 시험용으로 빼 둡니다.
시험 평점은 취향 벡터를 만들 때 쓰지 않습니다.


In [13]:
def split_by_user(ratings, test_size=0.2, random_state=42):
    """사용자마다 일부를 테스트로 빼고, 학습은 최소 1개를 남긴다."""
    frame = ratings.reset_index(drop=True)
    rng = np.random.default_rng(random_state)
    mask = np.zeros(len(frame), dtype=bool)
    for positions in frame.groupby("user_id", sort=False).indices.values():
        if len(positions) > 1:
            count = min(len(positions) - 1, max(1, int(len(positions) * test_size)))
            mask[rng.choice(positions, count, replace=False)] = True
    return frame.loc[~mask].copy(), frame.loc[mask].copy()


train, test = split_by_user(ratings)
print("학습:", len(train), "테스트:", len(test))
print("학습 사용자 수:", train["user_id"].nunique())


학습: 802553 테스트: 197656
학습 사용자 수: 6040


## 4. 사람 취향 벡터 만들기 (학습 평점만)

팀원과 같은 식입니다. 다만 **train만** 사용합니다.

1. `adjusted_rating = rating - 3`
2. 그 영화의 장르(0/1)에 이 값을 곱함
3. 사용자별로 더함
4. 길이를 1로 맞춤 (L2 정규화)

4점이면 그 장르 +1, 2점이면 그 장르 −1.


In [14]:
train = train.copy()
train["adjusted_rating"] = train["rating"] - 3.0

merged = train.merge(movie_feature_df, left_on="item_id", right_index=True)
for col in genre_cols:
    merged[col] = merged[col] * merged["adjusted_rating"]

user_profiles_raw = merged.groupby("user_id")[genre_cols].sum()
raw_values = user_profiles_raw.to_numpy(dtype=float)
norms = np.linalg.norm(raw_values, axis=1, keepdims=True)
norms[norms == 0] = 1.0
user_profiles_norm = raw_values / norms

user_ids = user_profiles_raw.index.to_numpy()
user_index = {int(u): i for i, u in enumerate(user_ids)}
print("유저 프로필 크기:", user_profiles_norm.shape)
user_profiles_raw.head()


유저 프로필 크기: (6040, 18)


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
user_id,,,,,,,,,,,,,,,,,,
1,6.0,5.0,13.0,18.0,14.0,2.0,0.0,26.0,3.0,0.0,0.0,15.0,0.0,4.0,4.0,2.0,4.0,0.0
2,19.0,11.0,0.0,0.0,13.0,7.0,0.0,53.0,0.0,0.0,0.0,0.0,1.0,13.0,5.0,7.0,9.0,4.0
3,19.0,22.0,3.0,3.0,20.0,0.0,0.0,7.0,3.0,0.0,-1.0,1.0,0.0,2.0,5.0,4.0,2.0,10.0
4,19.0,0.0,0.0,1.0,0.0,2.0,0.0,7.0,1.0,0.0,3.0,0.0,0.0,2.0,5.0,4.0,1.0,3.0
5,-10.0,0.0,4.0,4.0,20.0,7.0,3.0,-2.0,0.0,3.0,-2.0,2.0,2.0,-1.0,1.0,-5.0,3.0,1.0


In [23]:
user_profiles_norm_df = pd.DataFrame(
    user_profiles_norm,
    index=user_ids,
    columns=genre_cols,
)
user_profiles_norm_df.index.name = "user_id"
user_profiles_norm_df.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
user_id,,,,,,,,,,,,,,,,,,
1,0.144841,0.120701,0.313823,0.434524,0.337963,0.048280,0.000000,0.627646,0.072421,0.000000,0.000000,0.362103,0.000000,0.096561,0.096561,0.048280,0.096561,0.000000
2,0.306213,0.177281,0.000000,0.000000,0.209514,0.112815,0.000000,0.854172,0.000000,0.000000,0.000000,0.000000,0.016116,0.209514,0.080582,0.112815,0.145048,0.064466
3,0.495222,0.573415,0.078193,0.078193,0.521286,0.000000,0.000000,0.182450,0.078193,0.000000,-0.026064,0.026064,0.000000,0.052129,0.130322,0.104257,0.052129,0.260643
4,0.867227,0.000000,0.000000,0.045644,0.000000,0.091287,0.000000,0.319505,0.045644,0.000000,0.136931,0.000000,0.000000,0.091287,0.228218,0.182574,0.045644,0.136931
5,-0.391630,0.000000,0.156652,0.156652,0.783260,0.274141,0.117489,-0.078326,0.000000,0.117489,-0.078326,0.078326,0.078326,-0.039163,0.039163,-0.195815,0.117489,0.039163


한 명의 취향을 장르 이름과 같이 봅니다. 값이 큰 장르를 상대적으로 좋아한 것입니다.


In [15]:
example_user = int(user_ids[0])
profile = pd.Series(user_profiles_raw.loc[example_user], index=genre_cols)
print("user_id:", example_user)
print(profile.sort_values(ascending=False))
print()
print("이 사용자가 학습에서 준 평점 일부:")
train.loc[train["user_id"] == example_user, ["item_id", "rating"]].head()


user_id: 1
Drama          26.0
Children's     18.0
Musical        15.0
Comedy         14.0
Animation      13.0
Action          6.0
Adventure       5.0
Sci-Fi          4.0
Romance         4.0
War             4.0
Fantasy         3.0
Thriller        2.0
Crime           2.0
Documentary     0.0
Film-Noir       0.0
Horror          0.0
Mystery         0.0
Western         0.0
Name: 1, dtype: float64

이 사용자가 학습에서 준 평점 일부:


,item_id,rating
0,1193,5
1,661,3
2,914,3
5,1197,3
6,1287,5


## 5. 예측: 코사인 → 1~5점

사람 벡터와 영화 벡터가 둘 다 길이 1이면

`유사도 = np.dot(사람, 영화)`  ∈ [−1, 1]

3점이 기준이므로 평점은 이렇게 바꿉니다.

`점수 = clip(3 + 2 × 유사도, 1, 5)`

- 유사도 1 → 5점
- 유사도 0 → 3점
- 유사도 −1 → 1점

모르는 사용자/영화는 학습 평균으로 대체합니다.


In [16]:
global_mean = float(train["rating"].mean())
user_means = train.groupby("user_id")["rating"].mean().to_dict()
item_means = train.groupby("item_id")["rating"].mean().to_dict()
rating_min = float(train["rating"].min())
rating_max = float(train["rating"].max())


def predict(user_id, item_id):
    u = user_index.get(int(user_id))
    i = item_index.get(int(item_id))
    if u is not None and i is not None:
        sim = float(np.dot(user_profiles_norm[u], movie_multihot_norm[i]))
        score = 3.0 + 2.0 * sim
        return float(np.clip(score, rating_min, rating_max)), "content", 1
    if user_id in user_means:
        return float(user_means[user_id]), "user_mean", 0
    if item_id in item_means:
        return float(item_means[item_id]), "item_mean", 0
    return global_mean, "global_mean", 0


uid = 1
sample_item = int(test.loc[test["user_id"] == uid, "item_id"].iloc[0])
pred, method, n = predict(uid, sample_item)
actual = float(test.loc[(test["user_id"] == uid) & (test["item_id"] == sample_item), "rating"].iloc[0])
title = titles[item_index[sample_item]]
print(f"user={uid}, item={sample_item} ({title})")
print(f"실제 평점: {actual}, 예측: {pred:.3f}, method={method}")


user=1, item=3408 (Erin Brockovich (2000))
실제 평점: 4.0, 예측: 4.255, method=content


In [27]:
uid = 1
sample_item = int(test.loc[test["user_id"] == uid, "item_id"].iloc[0])

u = user_index[uid]
i = item_index[sample_item]

user_vec = user_profiles_norm[u]
item_vec = movie_multihot_norm[i]

calc = pd.DataFrame({
    "genre": genre_cols,
    "user_vec": user_vec,
    "item_vec": item_vec,
    "곱": user_vec * item_vec,
})
display(calc)

sim = float(np.dot(user_vec, item_vec))
raw_score = 3.0 + 2.0 * sim
score = float(np.clip(raw_score, 1, 5))

print("영화:", titles[i], "(item_id =", sample_item, ")")
print("내적(유사도) = 곱의 합 =", sim)
print("3 + 2 × 유사도 =", raw_score)
print("clip 후 예측 =", score)

,genre,user_vec,item_vec,곱
0,Action,0.144841,0.0,0.000000
1,Adventure,0.120701,0.0,0.000000
2,Animation,0.313823,0.0,0.000000
3,Children's,0.434524,0.0,0.000000
4,Comedy,0.337963,0.0,0.000000
5,Crime,0.048280,0.0,0.000000
6,Documentary,0.000000,0.0,0.000000
7,Drama,0.627646,1.0,0.627646
8,Fantasy,0.072421,0.0,0.000000
9,Film-Noir,0.000000,0.0,0.000000


영화: Erin Brockovich (2000) (item_id = 3408 )
내적(유사도) = 곱의 합 = 0.6276459144608478
3 + 2 × 유사도 = 4.255291828921695
clip 후 예측 = 4.255291828921695


## 6. 추천: 안 본 영화 중 점수 상위 K

학습에서 이미 평가한 영화는 빼고,
점수는 높은 순, 같으면 `item_id` 낮은 순입니다.


In [17]:
def recommend(user_id, top_n=10):
    seen = set(train.loc[train["user_id"] == user_id, "item_id"])
    records = []
    for item_id in item_ids:
        if int(item_id) in seen:
            continue
        score, method, n = predict(user_id, int(item_id))
        records.append((int(item_id), score, method, n))
    rec = pd.DataFrame(
        records,
        columns=["item_id", "predicted_rating", "method", "n_contributors"],
    )
    rec = rec.sort_values(["predicted_rating", "item_id"], ascending=[False, True])
    rec = rec.head(top_n).reset_index(drop=True)
    rec["title"] = rec["item_id"].map(lambda i: titles[item_index[i]])
    return rec


recommend(1, top_n=10)


,item_id,predicted_rating,method,n_contributors,title
0,34,4.616735,content,1,Babe (1995)
1,1014,4.616735,content,1,Pollyanna (1960)
2,1812,4.616735,content,1,Wide Awake (1998)
3,1288,4.533110,content,1,This Is Spinal Tap (1984)
4,241,4.502135,content,1,Fluke (1995)
5,262,4.502135,content,1,"Little Princess, A (1995)"
6,755,4.502135,content,1,Kim (1950)
7,917,4.502135,content,1,"Little Princess, The (1939)"
8,1012,4.502135,content,1,Old Yeller (1957)
9,1013,4.502135,content,1,"Parent Trap, The (1961)"


## 7. (선택) 작은 평가

테스트 앞 2000개만 MAE/RMSE를 봅니다.


In [18]:
sample_test = test.head(2000)
preds = [predict(u, i)[0] for u, i in sample_test[["user_id", "item_id"]].itertuples(index=False, name=None)]
err = np.asarray(preds) - sample_test["rating"].to_numpy()
print("n =", len(sample_test))
print("MAE =", float(np.abs(err).mean()))
print("RMSE =", float(np.sqrt(np.mean(err ** 2))))


n = 2000
MAE = 0.8741410700539225
RMSE = 1.1050941721048717


## 정리

| 단계 | 어디서 |
|---|---|
| 영화 장르 벡터 | 팀원 `movies_multihot_data.npz` |
| 평점 나누기 | 원본 `ratings.dat` → train / test |
| 사람 취향 벡터 | **train만**으로 여기서 계산 |
| 예측/추천 | 사람 벡터 · 영화 벡터 내적 → 1~5점 |

나중에 `src/content_based.py`의 `fit` / `predict` / `recommend`에 옮기면 됩니다.
